# CATMAID

A CATMAID project, and what asks it.

*Exported from Coda v0.0.0-test on 2026-01-01.*

In [ ]:
# pip install navis numpy pandas python-catmaid

import os
import pandas as pd
import numpy as np
import navis
import pymaid

In [ ]:
# Helpers, generated by Coda. These are the parts of the workflow that have
# no equivalent in the libraries this notebook imports, written out here so
# it stands on its own.

def coda_walk(seed_ids, direction, hops, one_hop):
    """Walk the connectome breadth-first from the seeds, as Coda's Connectivity node.

    `one_hop(ids, way)` returns the edges leaving `ids` in direction `way` ('downstream' or
    'upstream') with columns preId, preType, postId, postType and weight (ids as text).
    The result has the same columns plus `hop` and `direction`, each row oriented from
    presynaptic to postsynaptic.

    Notes:

    * Each neuron is expanded at most once, so recurrent loops do not repeat. Edges back
      into visited neurons are still reported.
    * An edge found again at a later hop keeps the hop and direction it was first found at.
    * direction="both" expands both ways at every hop, so it also finds neurons that share
      inputs with a seed.
    """
    columns = ['preId', 'preType', 'postId', 'postType', 'weight']
    ways = {'both': ['downstream', 'upstream'], 'inputs': ['upstream']}.get(direction, ['downstream'])
    frontier, expanded, seen = {str(i) for i in seed_ids}, set(), {}
    for hop in range(1, int(hops) + 1):
        todo = frontier - expanded
        if not todo:
            break
        expanded |= todo
        reached = set()
        for way in ways:
            for row in one_hop(todo, way).reindex(columns=columns).itertuples(index=False):
                key = (row.preId, row.postId)
                if key in seen:
                    # Found from both ends at the same hop: label it "both".
                    if seen[key][0] == hop and seen[key][1] != way:
                        seen[key][1] = 'both'
                    continue
                seen[key] = [hop, way, row]
                reached.add(row.postId if way == 'downstream' else row.preId)
        frontier = reached - expanded
    return pd.DataFrame([(*row, hop, way) for hop, way, row in seen.values()],
                        columns=columns + ['hop', 'direction'])


class CodaCatmaidDataset:
    """Coda's CATMAID dataset: one project on one CATMAID server, read through pymaid.

    `labels` is the neuron table Coda builds for the project, one row per skeleton: `neuronId`
    (the skeleton id, as text), `name`, `type`, `instance`, `ontologyId`, the remaining
    annotations joined with '; ', and the node count and cable length (nm). `instance` is the
    annotation meta-annotated "neuron name" and `type` the part of it before '#'; `ontologyId` is
    the one meta-annotated "Cell type". A project that meta-annotates no neuron names (L1) uses
    each neuron's name as its type.

    Every query names its CatmaidInstance (`remote_instance=` or `self.remote.fetch`), so two
    CATMAID datasets in one notebook never answer for each other. An empty set of neurons asks
    the server nothing and answers an empty table, as on the canvas.
    """

    TYPE_META = 'neuron name'
    ONTOLOGY_META = 'Cell type'

    def __init__(self, server, project_id, api_token=None):
        self.remote = pymaid.CatmaidInstance(server, api_token=api_token,
                                             project_id=int(project_id), make_global=False)
        self._labels = None
        self._type_map = None

    # ---- neurons ---------------------------------------------------------------------------

    def _vocabulary(self):
        """The names of the annotations meta-annotated "neuron name" and "Cell type", each an
        empty set where the project has no such meta-annotation.

        Asked of this server directly: pymaid's get_annotated looks the annotation up through
        the global instance, and on a missing one returns every entity on the server."""
        rm = self.remote
        listing = rm.fetch(rm._get_annotation_list())['annotations']
        ids = {a['name']: a['id'] for a in listing}

        def members(meta):
            if meta not in ids:
                return set()
            found = rm.fetch(rm._get_annotated_url(),
                             post={'annotated_with[0]': ids[meta], 'with_annotations': False})
            return {e['name'] for e in found['entities'] if e['type'] == 'annotation'}

        return members(self.TYPE_META), members(self.ONTOLOGY_META)

    def _summaries(self, skids):
        """Node count and cable length per skeleton, or none: a server that refuses this
        leaves the two columns empty rather than the table unbuilt, as on the canvas. pymaid
        has no wrapper for this endpoint."""
        url = self.remote.make_url(self.remote.project_id, 'skeletons', 'summary')
        post = {f'skeleton_ids[{i}]': s for i, s in enumerate(skids)}
        try:
            return self.remote.fetch(url, post=post)
        except Exception as error:
            print(f'No node counts or cable lengths: {error}')
            return {}

    @property
    def labels(self):
        if self._labels is None:
            rm = self.remote
            skids = sorted(int(s) for s in pymaid.get_skeleton_ids(remote_instance=rm))
            names = pymaid.get_names(skids, remote_instance=rm)
            annotations = pymaid.get_annotations(skids, remote_instance=rm)
            typed, ontology = self._vocabulary()
            summaries = self._summaries(skids)
            rows = []
            for skid in skids:
                key = str(skid)
                name = names.get(key)
                instance = term = None
                rest = []
                for label in annotations.get(key, []):
                    if label in typed:
                        instance = instance or label
                    elif label in ontology:
                        term = term or label
                    elif label not in rest:
                        rest.append(label)
                if instance is not None:
                    kind = instance.split('#', 1)[0]
                else:
                    kind = None if typed else name
                summary = summaries.get(key) or {}
                rows.append({
                    'neuronId': key,
                    'name': name,
                    'type': kind,
                    'instance': instance,
                    'ontologyId': term,
                    'annotations': '; '.join(rest) or None,
                    'nodes': summary.get('num_nodes'),
                    'cableLength': summary.get('cable_length'),
                })
            frame = pd.DataFrame(rows, columns=['neuronId', 'name', 'type', 'instance',
                                                'ontologyId', 'annotations', 'nodes',
                                                'cableLength'])
            frame['nodes'] = frame['nodes'].astype('Int64')
            self._labels = frame
        return self._labels

    def lookup(self, ids):
        """Return the neuron table rows for these skeleton ids."""
        wanted = {str(i) for i in ids}
        return self.labels[self.labels['neuronId'].isin(wanted)].reset_index(drop=True)

    def _types(self):
        if self._type_map is None:
            self._type_map = dict(zip(self.labels['neuronId'], self.labels['type']))
        return self._type_map

    # ---- connectivity ----------------------------------------------------------------------

    def _connected(self, queried, last):
        """Both directions' partners of these skeletons from one request, kept in `last` for
        the next call: a walk going both ways asks about the same skeletons twice per hop.
        `last` belongs to one walk, so the answer is not held after it.

        The `skeletons/connectivity` answer itself — sparse, partner to queried skeleton to five
        confidence counts — rather than pymaid.get_partners, which spreads it into a dense
        partners-by-queried frame and fetches every partner's name."""
        if last.get('ids') != queried:
            post = {f'source_skeleton_ids[{i}]': s for i, s in enumerate(queried)}
            post.update(boolean_op='OR', with_nodes=False)
            last['ids'] = queried
            last['answer'] = self.remote.fetch(self.remote._get_connectivity_url(), post=post)
        return last['answer']

    def _partners(self, ids, way, min_weight, keep, last):
        """One row per (queried skeleton, partner) with at least min_weight synapses, every
        confidence counted, oriented presynaptic to postsynaptic. With `keep`, only partners in
        it."""
        columns = ['preId', 'preType', 'postId', 'postType', 'weight']
        queried = sorted({int(i) for i in ids})
        if not queried:
            return pd.DataFrame(columns=columns)
        side = self._connected(queried, last).get('outgoing' if way == 'downstream' else 'incoming')
        types = self._types()
        rows = []
        for partner, entry in (side or {}).items():
            if keep is not None and partner not in keep:
                continue
            for query, counts in entry['skids'].items():
                # Every confidence counts: the weight is the sum of the five buckets.
                weight = sum(counts)
                if weight < max(1, min_weight):
                    continue
                pre, post = (query, partner) if way == 'downstream' else (partner, query)
                rows.append((pre, types.get(pre), post, types.get(post), weight))
        return pd.DataFrame(rows, columns=columns)

    def connectivity(self, seed_ids, direction, hops=1, min_weight=1, include_fragments=False):
        """Find partners hop by hop, as Coda's Connectivity node does.

        Without include_fragments, a partner must be in the neuron table (seeds are exempt):
        a project's synapses also reach skeletons its skeleton list does not hold.
        """
        keep = None if include_fragments else set(self.labels['neuronId']) | {str(i) for i in seed_ids}
        last = {}
        return coda_walk(seed_ids, direction, hops,
                         lambda ids, way: self._partners(ids, way, min_weight, keep, last))

    def adjacency(self, sources, targets, by_type=False):
        """Synapse counts from each source onto each target: the matrix (sources down, targets
        across, each once in the order given) and its non-zero cells as links. With by_type,
        lines are summed per type, and a neuron with no type keeps its own id."""
        rows = list(dict.fromkeys(str(i) for i in sources))
        cols = list(dict.fromkeys(str(i) for i in targets))
        counts = pd.DataFrame(0.0, index=rows, columns=cols)
        if rows and cols:
            found = pymaid.adjacency_matrix([int(i) for i in rows], [int(i) for i in cols],
                                            remote_instance=self.remote)
            found.index = found.index.astype(str)
            found.columns = found.columns.astype(str)
            counts = found.reindex(index=rows, columns=cols, fill_value=0).astype(float)
        types = self._types()
        # Each line's key, and the (id, type) it stands for: grouped, a typed neuron joins its
        # type's line, which has no id, and an untyped one keeps its own id and has no type.
        lines = {}
        for i in rows + cols:
            kind = types.get(i) if by_type else None
            lines[kind or i] = (None, kind) if kind else (i, types.get(i))
        if by_type:
            line = lambda i: types.get(i) or i
            counts = counts.groupby(counts.index.map(line), sort=False).sum()
            counts = counts.T.groupby(counts.columns.map(line), sort=False).sum().T
        counts.index.name = counts.columns.name = None
        cells = counts.stack()
        cells = cells[cells > 0]
        pre = [lines[k] for k in cells.index.get_level_values(0)]
        post = [lines[k] for k in cells.index.get_level_values(1)]
        links = pd.DataFrame({
            'preId': [p[0] for p in pre],
            'preType': [p[1] for p in pre],
            'postId': [p[0] for p in post],
            'postType': [p[1] for p in post],
            'weight': cells.astype(int).values,
        })
        return counts, links

    # ---- synapses --------------------------------------------------------------------------

    def _links(self, ids):
        """Pre- and postsynaptic connector links of these skeletons."""
        links = pymaid.get_connector_links([int(i) for i in ids], remote_instance=self.remote)
        return links[links['relation'].isin(['presynaptic_to', 'postsynaptic_to'])]

    def synapses(self, ids, polarity=None, min_confidence=0):
        """One row per connector link: neuronId, type, connectorId, polarity, confidence, x/y/z
        in nm. A connector is one presynaptic site, so this is one row per site."""
        if len(ids) == 0:
            return pd.DataFrame(columns=['neuronId', 'type', 'connectorId', 'polarity',
                                         'confidence', 'x', 'y', 'z'])
        links = self._links(ids)
        links = links.assign(polarity=links['relation'].map({'presynaptic_to': 'pre',
                                                             'postsynaptic_to': 'post'}))
        if polarity:
            links = links[links['polarity'] == polarity]
        if min_confidence > 0:
            links = links[links['confidence'] >= min_confidence]
        skids = links['skeleton_id'].astype(str)
        return pd.DataFrame({
            'neuronId': skids.values,
            'type': skids.map(self._types()).values,
            'connectorId': links['connector_id'].astype('int64').values,
            'polarity': links['polarity'].values,
            'confidence': links['confidence'].values,
            'x': links['x'].astype(float).values,
            'y': links['y'].astype(float).values,
            'z': links['z'].astype(float).values,
        })

    def _open_side(self, bound, relation, include_fragments):
        """Every skeleton on the other end of the connectors `bound` touches as `relation`."""
        if len(bound) == 0:
            return []
        links = self._links(bound)
        connectors = links.loc[links['relation'] == relation, 'connector_id'].unique().tolist()
        if not connectors:
            return []
        details = pymaid.get_connector_details(connectors, remote_instance=self.remote)
        if relation == 'presynaptic_to':
            found = [s for row in details['postsynaptic_to'] for s in row]
        else:
            found = details['presynaptic_to'].dropna().astype(int).tolist()
        if not include_fragments:
            published = set(self.labels['neuronId'])
            found = [s for s in found if str(s) in published]
        return found

    def synapses_between(self, sources=None, targets=None, location='pre', min_confidence=0,
                         include_fragments=True):
        """Every synapse from a source onto a target, one row per pre/post link pair on a
        connector, at the connector. neuronId is the presynaptic end; confidence is the drawn
        end's link. With only one side given, the other is every partner on its connectors —
        only those in the neuron table, unless include_fragments."""
        if sources is None and targets is None:
            raise ValueError('Synapses between two sets needs sources, targets or both.')
        if targets is None:
            targets = self._open_side(sources, 'presynaptic_to', include_fragments)
        elif sources is None:
            sources = self._open_side(targets, 'postsynaptic_to', include_fragments)
        if len(sources) == 0 or len(targets) == 0:
            return pd.DataFrame(columns=['neuronId', 'type', 'partnerId', 'partnerType',
                                         'polarity', 'confidence', 'x', 'y', 'z'])
        pairs = pymaid.get_connectors_between([int(i) for i in dict.fromkeys(sources)],
                                              [int(i) for i in dict.fromkeys(targets)],
                                              remote_instance=self.remote)
        if min_confidence > 0:
            pairs = pairs[(pairs['confidence1'] >= min_confidence)
                          & (pairs['confidence2'] >= min_confidence)]
        pre = pairs['source_neuron'].astype(str)
        post = pairs['target_neuron'].astype(str)
        drawn = 'post' if location == 'post' else 'pre'
        types = self._types()
        at = pairs['connector_loc'].tolist()
        return pd.DataFrame({
            'neuronId': pre.values,
            'type': pre.map(types).values,
            'partnerId': post.values,
            'partnerType': post.map(types).values,
            'polarity': drawn,
            'confidence': pairs['confidence2' if drawn == 'post' else 'confidence1'].values,
            'x': [p[0] for p in at],
            'y': [p[1] for p in at],
            'z': [p[2] for p in at],
        })

    # ---- geometry --------------------------------------------------------------------------

    def skeletons(self, ids):
        """The traced skeletons, as a navis NeuronList in nm. The soma is the node tagged
        'soma'; radii CATMAID leaves unset are -1."""
        if len(ids) == 0:
            return navis.NeuronList([])
        # A list even for one id: pymaid hands back a bare neuron then.
        return navis.NeuronList(pymaid.get_neurons([int(i) for i in ids], with_connectors=False,
                                                   remote_instance=self.remote))

    def volumes(self, names=None):
        """Neuropil meshes by name, as navis Volumes; every volume in the project if None."""
        if names is None:
            names = pymaid.get_volume(remote_instance=self.remote)['name'].tolist()
        found = pymaid.get_volume(list(names), remote_instance=self.remote)
        return list(found.values()) if isinstance(found, dict) else [found]


def coda_ids(df, *columns):
    """Cast id columns to text, so 18-digit neuron ids stay exact."""
    if df is None:
        return df
    for name in columns:
        if name not in df.columns:
            continue
        col = df[name]
        df[name] = (
            col.astype('Int64').astype('string')
            if pd.api.types.is_numeric_dtype(col)
            else col.astype('string')
        )
    return df


def coda_endpoint_neurons(connections, seeds=None):
    """List the neurons in an edge list: the seeds, then every partner, one row each.

    This is the Neuron Set output of Coda's Connectivity node. Seeds are always included,
    even those left without edges after min_weight. `seeds` is the seed table; its `type`
    column is used where it has one.
    """
    frames = []
    if seeds is not None:
        types = seeds['type'].values if 'type' in seeds.columns else None
        seeds = pd.DataFrame({'neuronId': seeds['neuronId'].values, 'type': types})
        frames.append(coda_ids(seeds, 'neuronId'))
    connections = coda_ids(connections.copy(), 'preId', 'postId')
    for id_col, type_col in (('preId', 'preType'), ('postId', 'postType')):
        frames.append(
            connections[[id_col, type_col]].rename(
                columns={id_col: 'neuronId', type_col: 'type'}
            )
        )

    rows = pd.concat(frames, ignore_index=True)
    rows['type'] = rows['type'].replace('', None)
    # Order by first appearance, but take the first non-empty type, which may come from
    # a later row (e.g. an untyped seed typed by an edge).
    typed = (
        rows.dropna(subset=['type'])
        .drop_duplicates(subset='neuronId')
        .set_index('neuronId')['type']
    )
    out = rows.drop_duplicates(subset='neuronId').reset_index(drop=True)
    out['type'] = out['neuronId'].map(typed)
    return out


import re

_CODA_OPERATORS = [("==", "eq"), ("!=", "ne"), (">=", "ge"), ("<=", "le"),
                   ("~", "match"), (">", "gt"), ("<", "lt"), ("=", "eq")]
_CODA_FIELD_NAME = re.compile(r"^[A-Za-z_][A-Za-z0-9_.]*$")


def _coda_tokenize(text):
    """Split on whitespace, keeping quoted text together."""
    tokens, i = [], 0
    while i < len(text):
        while i < len(text) and text[i].isspace():
            i += 1
        if i >= len(text):
            break
        start, quote = i, None
        while i < len(text):
            ch = text[i]
            if quote is not None:
                if ch == quote:
                    quote = None
            elif ch in "\"'":
                quote = ch
            elif ch.isspace():
                break
            i += 1
        tokens.append(text[start:i])
    return tokens


def _coda_unquote(value):
    if value[:1] in ("\"", "'") and len(value) >= 2:
        return value[1:-1] if value.endswith(value[0]) else value[1:]
    return value


def _coda_split_operator(token):
    """Split a token into (field, operator, value), or return None for a bare word.

    Longer operators are tried first so "!=" is not read as "=". The field must look like
    a column name, so "LC4-a" is not parsed as a comparison.
    """
    for symbol, op in _CODA_OPERATORS:
        at = token.find(symbol)
        if at <= 0:
            continue
        field = token[:at]
        if not _CODA_FIELD_NAME.match(field):
            continue
        return field, op, token[at + len(symbol):]
    return None


def _coda_parse_search(text):
    terms = []
    for raw in _coda_tokenize(text):
        negate = False
        if raw[:1] in ("!", "-") and len(raw) > 1 and _coda_split_operator(raw) is None:
            negate, raw = True, raw[1:]
        split = _coda_split_operator(raw)
        if split is None:
            value = _coda_unquote(raw)
            if value.startswith("/"):
                # A regex, as in neuroglancer. The closing slash is optional; a lone "/"
                # matches everything.
                pattern = value[1:]
                if pattern.endswith("/") and not pattern.endswith("\\/"):
                    pattern = pattern[:-1]
                if pattern:
                    # Not lowercased: the regex is matched case-insensitively instead,
                    # and lowercasing would change patterns like "[A-Z]".
                    terms.append(("regex", pattern, None, None, negate))
                continue
            if value:
                terms.append(("text", value.lower(), None, None, negate))
            continue
        field, op, value = split
        value = _coda_unquote(value)
        if not value:
            # A comparison without a value (e.g. "type=") is ignored.
            continue
        terms.append(("field", value, field, op, negate))
    return terms


def _coda_searchable(df):
    """Return the columns searched by free text and regex terms.

    Only text columns and neuronId, so "1200" finds a neuron id but not every neuron
    with 1200 synapses.
    """
    # Check both: pandas 3 gives text its own `str` dtype, which is not `object`.
    return [c for c in df.columns
            if pd.api.types.is_object_dtype(df[c]) or pd.api.types.is_string_dtype(df[c])
            or str(c) == "neuronId"]


def _coda_haystack(df):
    """Join the searchable columns into one lowercase string per row."""
    cols = _coda_searchable(df)
    if not cols:
        return pd.Series([""] * len(df), index=df.index)
    parts = [df[c].fillna("").astype(str) for c in cols]
    joined = parts[0]
    for part in parts[1:]:
        joined = joined.str.cat(part, sep=" ")
    return joined.str.lower()


def _coda_regex_mask(df, pattern):
    """Test whether any searchable field of a row matches a regex.

    Each field is tested on its own, so anchored patterns like "^LC4$" work. Missing
    values never match.
    """
    rx = re.compile(pattern, re.IGNORECASE)
    mask = pd.Series(False, index=df.index)
    for col in _coda_searchable(df):
        series = df[col]
        present = series.notna()
        text = series.astype(str).where(present, "")
        mask |= text.map(lambda v: rx.search(v) is not None) & present
    return mask


def _coda_field_mask(df, field, op, value):
    """Compare one column against a value.

    A missing value satisfies "!=" and nothing else, so status!=Traced returns both the
    untraced and the unlabelled neurons (unlike SQL, which would drop the unlabelled).
    """
    col = next((c for c in df.columns if str(c).lower() == field.lower()), None)
    if col is None:
        return pd.Series(False, index=df.index)
    series = df[col]
    missing = series.isna()

    if op == "match":
        # Unanchored, unlike neuPrint's "=~".
        rx = re.compile(value)
        found = series.fillna("").astype(str).map(lambda v: rx.search(v) is not None)
        return found & ~missing

    if pd.api.types.is_numeric_dtype(series):
        try:
            right = float(value)
        except ValueError:
            return pd.Series(False, index=df.index)
        left = pd.to_numeric(series, errors="coerce")
    else:
        right = value.lower()
        left = series.fillna("").astype(str).str.lower()

    if op == "eq":
        mask = left == right
    elif op == "ne":
        mask = left != right
    elif op == "gt":
        mask = left > right
    elif op == "lt":
        mask = left < right
    elif op == "ge":
        mask = left >= right
    else:
        mask = left <= right

    mask = mask.fillna(False).astype(bool)
    return (mask | missing) if op == "ne" else (mask & ~missing)


def coda_search(df, query):
    """Filter rows with Coda's Explore Dataset query language.

    Terms are combined with AND; a leading "!" or "-" negates a term. A bare word matches
    as a substring of any text column. A term starting with "/" is a regex tested against
    each text column, e.g. "/^LC4$". "field=value" compares one column; the other
    operators are ">", "<", ">=", "<=", "!=" and "~" (unanchored regex).

    Two differences from Coda, both of which can change which rows you get:

    * Matches come back in table order. Coda ranks them by relevance, which decides
      which rows are kept when the result is capped.
    * A query matching nothing returns nothing. Coda then retries it as a fuzzy match,
      so "mechnosensory" finds "mechanosensory" there but not here. Regex terms are not
      retried in Coda either.
    """
    terms = _coda_parse_search(query)
    if not terms:
        return df

    keep = pd.Series(True, index=df.index)
    haystack = None
    for kind, value, field, op, negate in terms:
        if kind == "text":
            if haystack is None:
                haystack = _coda_haystack(df)
            mask = haystack.str.contains(value, regex=False)
        elif kind == "regex":
            mask = _coda_regex_mask(df, value)
        else:
            mask = _coda_field_mask(df, field, op, value)
        keep &= ~mask if negate else mask

    return df[keep]

In [ ]:
# ── FAFB (CATMAID) ──
fafb_catmaid = CodaCatmaidDataset('https://catmaid-fafb.virtualflybrain.org', 1)

In [ ]:
# ── L1 (CATMAID) ──
l1_catmaid = CodaCatmaidDataset('https://l1em.catmaid.virtualflybrain.org', 1)

In [ ]:
# ── Custom CATMAID ──
# NOTE: A CATMAID server that needs an account takes an API token (your
# CATMAID profile page has one). Set CATMAID_API_TOKEN, or leave it unset
# for anonymous access.
custom_catmaid = CodaCatmaidDataset(
    'https://catmaid.example.org',
    3,
    api_token=os.environ.get('CATMAID_API_TOKEN'),
)

In [ ]:
# ── Find Neurons ──
# NOTE: CATMAID’s own name search matches substrings rather than patterns,
# so this reads the project’s neuron table once and filters it here, as Coda
# does.
find_neurons = fafb_catmaid.labels
find_neurons = find_neurons[(find_neurons['type'].notna() & (find_neurons['type'].astype(str) == 'Uniglomerular mALT DA1 lPN'))]

In [ ]:
# ── Explore Dataset ──
# NOTE: Explore Dataset searches the whole neuron table locally: here the
# CATMAID project’s, built from its annotations the first time anything
# needs it.
explore_dataset_all_ = fafb_catmaid.labels

explore_dataset_hits = coda_search(explore_dataset_all_, 'DA1')

# NOTE: Nothing is ticked on the canvas, so Selected is empty.
explore_dataset_selected = explore_dataset_all_.iloc[0:0]

In [ ]:
# ── Input IDs ──
_ids = [16, 27]
input_ids = fafb_catmaid.lookup(_ids)

In [ ]:
# ── ROI Meshes ──
roi_meshes = fafb_catmaid.volumes(['LAL_L', 'MB_PED_R'])

In [ ]:
# ── ROI Viewer ──
roi_viewer_meshes = {_volume.name: _volume for _volume in fafb_catmaid.volumes()}
print(f"{len(roi_viewer_meshes)} region meshes")

# Each value is a navis.Volume. To look at them: navis.plot3d(list(roi_viewer_meshes.values()))

In [ ]:
# ── Description ──
# NOTE: This card shows the project's description from the CATMAID server's
# project list, so there is no code to export.

In [ ]:
# ── Find Neurons ──
# NOTE: CATMAID’s own name search matches substrings rather than patterns,
# so this reads the project’s neuron table once and filters it here, as Coda
# does.
find_neurons_2 = l1_catmaid.labels
find_neurons_2 = find_neurons_2[(find_neurons_2['type'].notna() & find_neurons_2['type'].astype(str).str.contains('^(?:MBON-.*)$', regex=True, case=True))]

In [ ]:
# ── Explore Dataset ──
# NOTE: Explore Dataset searches the whole neuron table locally: here the
# CATMAID project’s, built from its annotations the first time anything
# needs it.
explore_dataset_2_all_ = custom_catmaid.labels

# NOTE: The search box is empty, so Hits is the whole table.
explore_dataset_2_hits = explore_dataset_2_all_

# NOTE: Nothing is ticked on the canvas, so Selected is empty.
explore_dataset_2_selected = explore_dataset_2_all_.iloc[0:0]

In [ ]:
# ── Connectivity ──
connectivity_connections = fafb_catmaid.connectivity(
    find_neurons['neuronId'].tolist(),
    direction='outputs',
    hops=1,
    min_weight=5,
    include_fragments=False,
)

_endpoints = coda_endpoint_neurons(connectivity_connections, find_neurons)
connectivity_neuron_set = _endpoints[['neuronId']].merge(fafb_catmaid.labels, on='neuronId')

In [ ]:
# ── Adjacency ──
adjacency_matrix, adjacency_links = fafb_catmaid.adjacency(
    find_neurons['neuronId'].tolist(),
    find_neurons['neuronId'].tolist(),
    by_type=False,
)

In [ ]:
# ── Adjacency ──
adjacency_2_matrix, adjacency_2_links = fafb_catmaid.adjacency(
    find_neurons['neuronId'].tolist(),
    find_neurons['neuronId'].tolist(),
    by_type=True,
)

In [ ]:
# ── Synapses Between ──
synapses_between = fafb_catmaid.synapses_between(
    sources=find_neurons['neuronId'].tolist(),
    targets=find_neurons['neuronId'].tolist(),
    location='post',
    min_confidence=3,
)

In [ ]:
# ── Connectivity ──
connectivity_2_connections = fafb_catmaid.connectivity(
    input_ids['neuronId'].tolist(),
    direction='both',
    hops=2,
    min_weight=20,
    include_fragments=True,
)

connectivity_2_neuron_set = coda_endpoint_neurons(connectivity_2_connections, input_ids)

In [ ]:
# ── Skeletons ──
# NOTE: CATMAID leaves a radius it was not given at -1, which pymaid keeps;
# the canvas draws those nodes at zero radius. Cable length and the tree are
# the same.
skeletons = fafb_catmaid.skeletons(input_ids['neuronId'].head(10000).astype('int64').tolist())

In [ ]:
# ── Synapses ──
synapses = fafb_catmaid.synapses(
    input_ids['neuronId'].tolist(),
    polarity=None,
)

In [ ]:
# ── Synapses ──
synapses_2 = fafb_catmaid.synapses(
    input_ids['neuronId'].tolist(),
    polarity='pre',
    min_confidence=4,
)

In [ ]:
# ── Synapses Between ──
synapses_between_2 = fafb_catmaid.synapses_between(
    sources=input_ids['neuronId'].tolist(),
    targets=None,
    location='pre',
    include_fragments=False,
)